# Andrea Pezzolla & Matteo Maruca
Progetto 2 DataScience, SUPSI, 2026

In [ ]:
# Import
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [ ]:
# Dataframe
df = pd.read_csv("100097.csv", sep=';')

In [ ]:
# TRADUZIONI COLONNE 

# ── Rinomina colonne in italiano ──────────────────────────────
df = df.rename(columns={
    "Timestamp"                         : "timestamp",
    "Messung-ID"                        : "id_postazione",
    "Richtung ID"                       : "id_direzione",
    "Geschwindigkeit"                   : "velocita_kmh",
    "Zeit"                              : "ora",
    "Datum"                             : "data",
    "Datum und Zeit"                    : "data_ora",
    "Messbeginn"                        : "inizio_misurazione",
    "Messende"                          : "fine_misurazione",
    "Zone"                              : "zona_limite",
    "Ort"                               : "localita",
    "Richtung"                          : "direzione",
    "Koordinaten"                       : "coordinate",
    "Übertretungsquote"                 : "tasso_infrazioni_pct",
    "Geschwindigkeit V50"               : "v50",
    "Geschwindigkeit V85"               : "v85",
    "Strasse"                           : "via",
    "Hausnummer"                        : "numero_civico",
    "Fahrzeuge"                         : "n_veicoli",
    "Fahrzeuglänge"                     : "lunghezza_veicolo_m",
    "Kennzahlen pro Mess-Standort"      : "link_statistiche",
    "geo_point_2d"                      : "geo_point",
})

# ── Conversioni di tipo ───────────────────────────────────────
df["timestamp"]           = pd.to_datetime(df["timestamp"], utc=True)
df["inizio_misurazione"]  = pd.to_datetime(df["inizio_misurazione"])
df["fine_misurazione"]    = pd.to_datetime(df["fine_misurazione"])

# Estrai colonne utili da timestamp
df["anno"]         = df["timestamp"].dt.year
df["mese"]         = df["timestamp"].dt.month
df["giorno_sett"]  = df["timestamp"].dt.day_name(locale="it_IT.UTF-8")  # es. "Lunedì"
df["ora_intera"]   = df["timestamp"].dt.hour

# Zona limite come categoria
#df["zona_limite"] = df["zona_limite"].astype(int).astype(str).apply(lambda x: f"Zona {x}")

print("Colonne rinominate:")
print(df.columns.tolist())
print(f"\nDimensioni: {df.shape[0]:,} righe × {df.shape[1]} colonne")
print(df.dtypes)

In [ ]:
df["zona_limite"].unique().max()

In [ ]:
# Top 15 postazioni per tasso medio di infrazioni

# Aggiungiamo 'zona_limite' al groupby per conservare il dato
top_infrazioni = (
    df.groupby(["localita", "via", "zona_limite"])["tasso_infrazioni_pct"]
    .mean()
    .reset_index()
    .sort_values("tasso_infrazioni_pct", ascending=False)
    .head(15)
)

fig2 = px.bar(
    top_infrazioni,
    x="tasso_infrazioni_pct",
    y="via",
    orientation="h",
    color="tasso_infrazioni_pct",
    color_continuous_scale="Teal", 
    title="<b>Top 15 Postazioni per Tasso di Infrazioni</b>",
    labels={
        "tasso_infrazioni_pct": "Tasso infrazioni (%)",
        "via":                  "Via",
        "localita":             "Località",
        "zona_limite":          "Limite (km/h)"
    },
    # Configurazione dettagliata dell'hover
    hover_data={
        "via": False, # Nasconde la via dal box perché è già sull'asse Y
        "localita": True,
        "zona_limite": True,
        "tasso_infrazioni_pct": ":.2f" # Formatta il numero con 2 decimali (es. 12.34 al posto di 12.34567)
    }
)

fig2.update_layout(
    yaxis={"categoryorder": "total ascending"},
    coloraxis_showscale=False,
    plot_bgcolor="white", # Sfondo bianco per una lettura più pulita
)

# Aggiungiamo la griglia verticale leggera per facilitare la lettura dei valori
fig2.update_xaxes(showgrid=True, gridcolor="#E5E5E5")

fig2.show()


In [ ]:
# Zoom sulla peggiore postazione: Distribuzione velocità

# 1. Estraiamo automaticamente il nome della via e il limite dal primo in classifica
via_peggiore = top_infrazioni.iloc[0]["via"]
limite_peggiore = top_infrazioni.iloc[0]["zona_limite"]

# 2. Filtriamo il dataframe originale 'df' per prendere solo i dati di quella via
df_peggiore = df[df["via"] == via_peggiore]

# 3. Creiamo l'istogramma (la "campana") delle velocità misurate
fig3 = px.histogram(
    df_peggiore,
    x="velocita_kmh",
    nbins=30, # Regola questo numero per fare le barre più o meno spesse
    title=f"<b>Distribuzione delle Velocità - {via_peggiore}</b>",
    labels={
        "velocita_kmh": "Velocità Misurata (km/h)",
        "count": "Numero di Veicoli"
    },
    color_discrete_sequence=["#1f77b4"],
    opacity=0.8
)

# 4. Aggiungiamo la riga verticale rossa per indicare dove inizia l'infrazione
fig3.add_vline(
    x=limite_peggiore, 
    line_width=3, 
    line_dash="dash", 
    line_color="red",
    annotation_text=f"Limite ({limite_peggiore} km/h)", 
    annotation_position="top right",
    annotation_font_color="red",
    annotation_font_size=14
)

# 5. Pulizia del layout
fig3.update_layout(
    plot_bgcolor="white",
    bargap=0.05, # Piccolo spazio tra le barre per renderle più leggibili
    yaxis_title="Numero di Veicoli"
)

# Aggiungiamo griglie leggere
fig3.update_xaxes(showgrid=True, gridcolor="#E5E5E5")
fig3.update_yaxes(showgrid=True, gridcolor="#E5E5E5")

fig3.show()

In [ ]:
#Heatmap - nr. misurazioni per giorno e ora (media giornaliera)

# Ordine corretto dei giorni
ordine_giorni = ["Lunedì", "Martedì", "Mercoledì", "Giovedì", "Venerdì", "Sabato", "Domenica"]

# 1. Contiamo i veicoli (numero di righe) per ogni SINGOLA DATA e ORA
df_volumi_giornalieri = df.groupby(["data", "giorno_sett", "ora_intera"]).size().reset_index(name="veicoli_orari")

# 2. Calcoliamo la MEDIA di questi volumi per avere il dato tipico "per giorno"
pivot_traffico = (
    df_volumi_giornalieri.groupby(["giorno_sett", "ora_intera"])["veicoli_orari"]
    .mean()
    .reset_index()
    .pivot(index="giorno_sett", columns="ora_intera", values="veicoli_orari")
    .reindex(ordine_giorni)
)

# 3. Creiamo la Heatmap con il nuovo Titolo e Sottotitolo
fig3 = px.imshow(
    pivot_traffico,
    color_continuous_scale="inferno", 
    aspect="auto",
    title="<b>Misurazioni/Giorno per Giorno e Ora</b><br><sup style='font-size:14px; font-weight:normal; color:gray'>Numero medio giornaliero di veicoli rilevati all'ora su singola giornata</sup>",
    labels={
        "x":     "Ora del giorno",
        "y":     "Giorno",
        "color": "Rilevazioni"
    }
)

# Aggiornamento dei font e del design generale
fig3.update_layout(
    title=dict(font=dict(size=20)),
    xaxis_title="<b>Ora del giorno</b>",
    yaxis_title="",
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(t=80) # Diamo respiro al sottotitolo
)

# Forza la visualizzazione di tutte le ore (da 0 a 23)
fig3.update_xaxes(
    tickmode="linear", 
    tick0=0, 
    dtick=1,
    showgrid=False
)

fig3.update_yaxes(showgrid=False)

# Tooltip personalizzato aggiornato
fig3.update_traces(
    hovertemplate="<b>%{y}</b> alle ore <b>%{x}:00</b><br>Media rilevazioni: %{z:,.0f} veicoli<extra></extra>"
)

fig3.show()

In [ ]:
#Tasso Infrazioni per Giorno e Ora - Heatmap

df["infrazioni_assolute"] = (df["tasso_infrazioni_pct"] / 100) * df["n_veicoli"]
df["infrazioni_assolute"] = df["infrazioni_assolute"].fillna(0)

df_raggruppato = df.groupby(["giorno_sett", "ora_intera"])[["infrazioni_assolute", "n_veicoli"]].sum().reset_index()

df_raggruppato["tasso_medio_reale_pct"] = (df_raggruppato["infrazioni_assolute"] / df_raggruppato["n_veicoli"]) * 100

# Ordine corretto dei giorni
ordine_giorni = ["Lunedì", "Martedì", "Mercoledì", "Giovedì", "Venerdì", "Sabato", "Domenica"]

pivot_inf = (
    df_raggruppato.pivot(index="giorno_sett", columns="ora_intera", values="tasso_medio_reale_pct")
    .reindex(ordine_giorni)
)

fig4 = px.imshow(
    pivot_inf,
    color_continuous_scale="inferno",
    aspect="auto",
    title="<b>Tasso Infrazioni per Giorno e Ora (%)</b><br><sup style='font-size:13px; font-weight:normal; color:gray'>Calcolo del tasso ponderato sui volumi di traffico effettivi</sup>",
    labels={
        "x":     "Ora del giorno",
        "y":     "Giorno",
        "color": "Infrazioni (%)"
    }
)

fig4.update_layout(
    title=dict(font=dict(size=20)),
    xaxis_title="<b>Ora del giorno</b>",
    yaxis_title="",
    plot_bgcolor="white",
    paper_bgcolor="white",
)

# Forza la visualizzazione di tutte le ore (da 0 a 23)
fig4.update_xaxes(
    tickmode="linear", 
    tick0=0, 
    dtick=1,
    showgrid=False
)

fig4.update_yaxes(showgrid=False)

fig4.update_traces(
    hovertemplate="<b>%{y}</b> alle ore <b>%{x}:00</b><br>Tasso infrazioni: %{z:.2f}%<extra></extra>"
)

fig4.show()

In [ ]:
# SubPlot – media annua infrazioni veicoli

# 1. Categorie veicoli
bins   = [0, 2.5, 5.0, 7.0, 100.0]
labels = ["Moto", "Auto", "Furgoni", "Camion/Bus"]

df["cat_veicolo"] = pd.cut(df["lunghezza_veicolo_m"], bins=bins, labels=labels, right=True)

# Calcolo anni totali per la media annua
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
anni_totali = df["timestamp"].dt.year.nunique()
if anni_totali == 0: anni_totali = 1

# Pulizia zone 
df_clean = df.dropna(subset=["zona_limite"]).copy()
df_clean["zona_limite_str"] = df_clean["zona_limite"].apply(lambda x: str(int(x)))

ordine_zone = sorted(df_clean["zona_limite"].unique())
ordine_zone_str = [str(int(z)) for z in ordine_zone]

# 2. Calcolo delle infrazioni reali per riga
df_clean["infrazione_reale"] = df_clean["tasso_infrazioni_pct"] / 100

# 3. Raggruppamento e calcolo della MEDIA ANNUA
df_infrazioni = (
    df_clean.groupby(["zona_limite_str", "cat_veicolo"], observed=False)["infrazione_reale"]
    .sum()
    .reset_index(name="Totale_Infrazioni")
)

df_infrazioni["Media_Annua"] = df_infrazioni["Totale_Infrazioni"] / anni_totali

# 4. Calcolo Percentuali
totale_media_zona = df_infrazioni.groupby("zona_limite_str", observed=False)["Media_Annua"].transform("sum")
df_infrazioni["Percentuale"] = (df_infrazioni["Media_Annua"] / totale_media_zona * 100).fillna(0)

# Palette colori
colori_veicoli = {
    "Moto": "#00B4D8",        
    "Auto": "#E07B39",        
    "Furgoni": "#2A9D8F",     
    "Camion/Bus": "#D62828"   
}

# 5. Definizione del SubPlot
fig6 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Media Annua Infrazioni", "Distribuzione % delle Infrazioni"),
    shared_yaxes=True,
    column_widths=[0.60, 0.40]
)

# --- GRAFICO A SINISTRA - barre orizzontali raggruppate ---
for cat in reversed(labels): 
    subset = df_infrazioni[df_infrazioni["cat_veicolo"] == cat]
    
    fig6.add_trace(
        go.Bar(
            x=subset["Media_Annua"],
            y=subset["zona_limite_str"],
            name=cat,
            orientation='h',
            marker_color=colori_veicoli[cat],
            legendgroup=cat,
            offsetgroup=cat, 
            hovertemplate='<b>Zona %{y}</b><br>%{x:,.0f} infrazioni/anno<extra>' + cat + '</extra>'
        ),
        row=1, col=1
    )

# --- GRAFICO A DESTRA - barre orizzontali in pila al 100% ---
cumsum = {z: 0.0 for z in ordine_zone_str}

for cat in labels:
    subset = df_infrazioni[df_infrazioni["cat_veicolo"] == cat].set_index("zona_limite_str")

    x_vals    = [subset.loc[z, "Percentuale"] if z in subset.index else 0.0 for z in ordine_zone_str]
    base_vals = [cumsum[z] for z in ordine_zone_str]

    for z, x in zip(ordine_zone_str, x_vals):
        cumsum[z] += x

    fig6.add_trace(
        go.Bar(
            x=x_vals,
            y=ordine_zone_str,
            base=base_vals,             
            name=cat,
            orientation='h',
            marker_color=colori_veicoli[cat],
            legendgroup=cat,
            width=0.75,       
            offset=-0.375,    
            showlegend=False, 
            hovertemplate='<b>Zona %{y}</b><br>%{x:.1f}% del totale<extra>' + cat + '</extra>'
        ),
        row=1, col=2
    )

# 6. Layout finale
fig6.update_layout(
    barmode='group',
    bargap=0.15,          
    bargroupgap=0.02,     
    title="<b>Infrazioni/Anno per Categoria Veicolo e Limite Velocità</b>",
    legend_title="<b>Categoria Veicolo</b>",
    legend_traceorder="reversed",
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=550,
    margin=dict(t=80, b=40, l=60, r=20)
)

# Linee di separazione orizzontali
for i in range(len(ordine_zone_str) - 1):
    fig6.add_hline(
        y=i + 0.5, 
        line_dash="solid",    
        line_color="#E0E0E0", 
        line_width=1.5,
        layer="below",        
        row='all', col='all'  
    )

# Assi X
fig6.update_xaxes(title_text="Infrazioni / Anno", showgrid=True, gridcolor="#E5E5E5", row=1, col=1)
fig6.update_xaxes(title_text="% Infrazioni", range=[0, 100], showgrid=True, gridcolor="#E5E5E5", ticksuffix="%", row=1, col=2)

# Asse Y
fig6.update_yaxes(
    title_text="<b>Zona</b>", 
    categoryorder='array', 
    categoryarray=ordine_zone_str[::-1], 
    row=1, col=1
)

fig6.show()

In [ ]:
# Mappa interattiva delle postazioni (Plotly mapbox)

df[["lat", "lon"]] = df["geo_point"].str.split(",", expand=True).astype(float)

df["infrazioni_assolute"] = (df["tasso_infrazioni_pct"] / 100) * df["n_veicoli"]
df["infrazioni_assolute"] = df["infrazioni_assolute"].fillna(0)

mappa = (
    df.groupby(["id_postazione", "via", "zona_limite", "lat", "lon"])
    .agg(
        tot_infrazioni = ("infrazioni_assolute", "sum"),
        tot_veicoli    = ("n_veicoli", "sum"),
        vel_media      = ("velocita_kmh", "mean"),
        n_rilevazioni  = ("n_veicoli", "count") # Conta quante righe (misurazioni) abbiamo per questo radar
    )
    .reset_index()
)
mappa["infrazioni_pct"] = (mappa["tot_infrazioni"] / mappa["tot_veicoli"]) * 100

# Normalizziamo il volume (Grandezza pallino) per non penalizzare i radar accesi per meno tempo
mappa["veicoli_medi"] = mappa["tot_veicoli"] / mappa["n_rilevazioni"]

fig8 = px.scatter_map(
    mappa,
    lat="lat",
    lon="lon",
    color="infrazioni_pct",
    size="veicoli_medi",
    size_max=25, # Aumentato leggermente per rendere più visibili i pallini
    hover_name="via",
    hover_data={
        "lat": False,
        "lon": False,
        "infrazioni_pct": ":.2f", 
        "vel_media": ":.1f", 
        "zona_limite": True,
        "veicoli_medi": ":.0f"
    },
    color_continuous_scale="inferno",
    zoom=12,
    map_style="carto-positron",
    title="<b>Mappa Postazioni – Tasso di Infrazioni vs Volume di Traffico</b>",
    labels={
        "infrazioni_pct": "Infrazioni (%)",
        "vel_media":      "Vel. media (km/h)",
        "veicoli_medi":   "Veicoli (media/ril.)",
        "zona_limite":    "Limite"
    }
)

fig8.update_layout(
    margin=dict(l=0, r=0, t=50, b=0),
    title=dict(font=dict(size=20))
)

fig8.show()

# Grafici Maru

## Grafico scatter geo che mostra l'omogeneità delle registrazioni nel canton basilea

In [ ]:
df[["lat", "lon"]] = (
    df["geo_point"]
    .str.split(",", expand=True)
    .astype(float)
)

#df['zona_limite'] = df['zona_limite'].str.extract(r'(\d+)')[0].astype(int)

df['Superamento'] = df['velocita_kmh'] - df['zona_limite']
idx_max = df.groupby(['lat', 'lon'])['Superamento'].idxmax()
df_peggiori = df.loc[idx_max].reset_index(drop=True)

fig = px.scatter_map(
    df_peggiori,
    lat="lat",
    lon="lon",
    zoom=12,                                 # Livello di zoom ideale per una città/cantone
    hover_name='zona_limite',
    center={"lat": 47.5596, "lon": 7.5886},  # Coordinate del centro di Basilea
    title="Misurazioni di Velocità - superamento limite kmh",
    size="Superamento",
    color="velocita_kmh",
    color_continuous_scale="Reds"
)

fig.update_layout(map_style="carto-positron")

fig.show()

IL grafico mostra il superamento massimo registrato da ogni radar e si vede bene perche con il colore vediamo la velocita a cui andava e invece con la grandezza quanto ha superato il limite.

**TODO**
Mappa interattiva 
Trend temporale 
Correlazione meteo 
Differenza velocità – tipo di veicolo (moto/camion)


## Numero superamenti per giorno della settimana

In [ ]:
df['superamento'] = df['velocita_kmh'] - df['zona_limite']

# Uso .copy() per evitare il SettingWithCopyWarning di Pandas
df_superamento = df[df['superamento'] > 0].copy()

df_superamento['data'] = pd.to_datetime(df_superamento['data'], format='%d.%m.%y', errors='coerce')

# Mappatura dal numero del giorno al nome in italiano
dizionario_giorni = {
    0: 'Lunedì', 
    1: 'Martedì', 
    2: 'Mercoledì', 
    3: 'Giovedì', 
    4: 'Venerdì', 
    5: 'Sabato', 
    6: 'Domenica'
}

# Estraiamo il giorno della settimana (da 0 a 6) e lo traduciamo con il dizionario
df_superamento['giorno_settimana'] = df_superamento['data'].dt.dayofweek.map(dizionario_giorni)

# Lista per forzare l'ordine logico (e non alfabetico) nel grafico
ordine_giorni_it = ['Lunedì', 'Martedì', 'Mercoledì', 'Giovedì', 'Venerdì', 'Sabato', 'Domenica']

fig = px.histogram(
    df_superamento,
    y='giorno_settimana',
    category_orders={"giorno_settimana": ordine_giorni_it},
    text_auto=True,
    title="Numero di infrazioni per giorno della settimana",
    labels={'giorno_settimana': 'Giorno della Settimana', 'count': 'Numero di Infrazioni'}
)

fig.show()

C'è una tendenza a fare infrazioni minore nel weekend rispetto al resto della settimana

In [ ]:
df['zona_limite'].unique()

In [ ]:
df[["lat", "lon"]] = (
    df["geo_point"]
    .str.split(",", expand=True)
    .astype(float)
)

df_notte_alto_limite = df[
    (df['ora_intera'].isin([22, 23, 0, 1, 2, 3, 4, 5])) & 
    (df['zona_limite'] >= 50) &
    (df['velocita_kmh'] > df['zona_limite'])
].copy()

df_notte_alto_limite['superamento'] = df_notte_alto_limite['velocita_kmh'] - df_notte_alto_limite['zona_limite']
idx_max = df_notte_alto_limite.groupby(['lat', 'lon'])['superamento'].idxmax()

df_peggiore_notte = df_notte_alto_limite.loc[idx_max]

fig = px.scatter_map(
    df_peggiore_notte,
    lat="lat",
    lon="lon",
    size="zona_limite",
    color="superamento",
    color_continuous_scale="Inferno",
    hover_name="superamento",
    hover_data=["velocita_kmh", "zona_limite", "ora", "data"],
    zoom=12,
    center={"lat": 47.5596, "lon": 7.5886},
    map_style="carto-positron",
    title="Pirati della Notte: Infrazioni su Strade ad Alto Limite (>= 50 km/h, 22:00 - 05:00)"
)

fig.update_layout(margin={"r":0, "t":40, "l":0, "b":0})

fig.show()

In [ ]:


# 1. Assicuriamoci che la colonna delle date sia in formato datetime
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

# 2. Filtriamo solo i dati dal 2024 in poi
df_recent = df[df['timestamp'].dt.year >= 2024].copy()

# 3. Creiamo una colonna "Anno-Mese" (es. '2024-01', '2024-02') per l'asse X
df_recent['mese_anno'] = df_recent['timestamp'].dt.strftime('%Y-%m')

# 4. Raggruppiamo per Radar e Mese, calcolando il totale dei veicoli passati
# (Questo è il nostro resampling/raggruppamento temporale manuale)
df_grouped = df_recent.groupby(['id_postazione', 'mese_anno'])['n_veicoli'].sum().reset_index()

# 5. Creiamo la Pivot Table per la Heatmap
# Le colonne mancanti (mesi in cui il radar era spento) diventeranno NaN.
# Usiamo fillna(0) per riempire quei buchi con degli zeri.
pivot_heatmap = df_grouped.pivot(index='id_postazione', columns='mese_anno', values='n_veicoli').fillna(0)

# 6. Trasformiamo i dati in Binari (0 o 1) come hai richiesto
# Se i veicoli sono > 0 diventa 1 (Nero/Acceso), altrimenti 0 (Bianco/Spento)
pivot_binaria = (pivot_heatmap > 0).astype(int)

# 7. Disegniamo la Heatmap
fig_uptime = px.imshow(
    pivot_binaria,
    color_continuous_scale=["white", "black"], # 0 = bianco, 1 = nero
    aspect="auto",
    title="<b>Verifica Attività Radar dal 2024</b><br><sup style='font-size:13px; color:gray'>Quadrato nero: Radar acceso (rilevato traffico) | Quadrato bianco: Nessun dato (Radar spento/offline)</sup>",
    labels={
        "x": "Mese",
        "y": "ID Radar (Postazione)",
        "color": "Stato"
    }
)

# 8. Miglioramenti estetici
fig_uptime.update_layout(
    xaxis_title="<b>Mese e Anno</b>",
    yaxis_title="<b>ID Postazione</b>",
    coloraxis_showscale=False, # Nascondiamo la legenda laterale perché è solo bianco/nero
    plot_bgcolor="lightgray",  # Sfondo grigio per far risaltare i quadrati bianchi
    paper_bgcolor="white"
)

# Forziamo Plotly a mostrare tutte le etichette per non saltare mesi o radar
fig_uptime.update_xaxes(type='category', tickangle=45)
fig_uptime.update_yaxes(type='category')

# Tooltip personalizzato per mostrare al passaggio del mouse se è acceso o spento
fig_uptime.update_traces(
    hovertemplate="<b>Radar %{y}</b><br>Mese: %{x}<br>Stato: %{z}<extra></extra>"
)

fig_uptime.show()